In [0]:
%python
from datetime import date, timedelta

CAT     = "dev_insurance"
RAW     = f"/Volumes/{CAT}/landing/raw"
INBOUND = f"/Volumes/{CAT}/landing/inbound"
START   = date(2026, 1, 1)

def _copy(src_path, dest_dir):
    dbutils.fs.mkdirs(dest_dir)
    fname = src_path.split("/")[-1]
    try:
        dbutils.fs.cp(src_path, f"{dest_dir}/{fname}")
        print(f"  landed {fname}")
    except Exception as e:
        if "FileNotFound" in str(e) or "does not exist" in str(e).lower():
            print(f"  (no file) {fname}")      # e.g. day 15 bordereau is 'late'
        else:
            raise

def land_day(day: int):
    """Simulate the files that arrive on day N (1-30)."""
    ds = (START + timedelta(days=day - 1)).strftime("%Y%m%d")
    print(f"Day {day} ({ds})")
    if day == 1:   # reference data arrives once
        _copy(f"{RAW}/reference/ref_product.csv", f"{INBOUND}/ref_product")
        _copy(f"{RAW}/reference/ref_broker.csv",  f"{INBOUND}/ref_broker")
    _copy(f"{RAW}/policy_admin/policyholder/policyholder_{ds}.csv", f"{INBOUND}/policyholder")
    _copy(f"{RAW}/policy_admin/policy/policy_{ds}.csv",             f"{INBOUND}/policy")
    _copy(f"{RAW}/bordereaux/premium/premium_bordereau_{ds}.csv",   f"{INBOUND}/premium_bordereau")
    _copy(f"{RAW}/claims/claims_events_{ds}.json",                  f"{INBOUND}/claim_events")

def land_late_file():
    """Day 15's bordereau finally arrives (run this on 'day 17')."""
    _copy(f"{RAW}/bordereaux/late/premium_bordereau_20260115.csv", f"{INBOUND}/premium_bordereau")

In [0]:
%python
land_day(2)

In [0]:
%python
# notebook 03_land_files
for d in range(18, 30):
    land_day(d)

In [0]:
%python
land_day(16)

In [0]:
%python
land_late_file()